In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

from functions import compress_channel

In [4]:
# SVD_with_SymPy.ipynb
# You can copy each section into separate Jupyter notebook cells.

# =========================
# Cell 1: Import SymPy
# =========================

import sympy as sp
from sympy import Matrix, pprint

sp.init_printing()


# =========================
# Cell 2: Input Matrix
# =========================

# Example matrix
A = Matrix([
    [1, 2, 3, 4],
    [1, 2, 3, 4],
    [1, 2, 3, 5]
])

print("Matrix A:")
pprint(A)


# =========================
# Cell 3: Compute SVD
# =========================

# Singular Value Decomposition
U, S, V = A.singular_value_decomposition()

print("\nMatrix U:")
pprint(U)

print("\nMatrix S:")
pprint(S)

print("\nMatrix V:")
pprint(V)


# =========================
# Cell 4: Verify Decomposition
# =========================

# Reconstruct A
A_reconstructed = U * S * V.T

print("\nReconstructed A:")
pprint(sp.simplify(A_reconstructed))


# =========================
# Cell 5: Optional Numerical Approximation
# =========================

print("\nNumerical approximation:\n")

print("U ≈")
pprint(U.evalf())

print("\nS ≈")
pprint(S.evalf())

print("\nV ≈")
pprint(V.evalf())

Matrix A:
⎡1  2  3  4⎤
⎢          ⎥
⎢1  2  3  4⎥
⎢          ⎥
⎣1  2  3  5⎦

Matrix U:
⎡         21    √9689                   21    √9689        ⎤
⎢         ─── - ─────                   ─── + ─────        ⎥
⎢         136    136                    136    136         ⎥
⎢─────────────────────────────  ───────────────────────────⎥
⎢     ________________________       ______________________⎥
⎢    ╱                  2           ╱                    2 ⎥
⎢   ╱    ⎛  21    √9689⎞           ╱        ⎛21    √9689⎞  ⎥
⎢  ╱   2⋅⎜- ─── + ─────⎟  + 1     ╱   1 + 2⋅⎜─── + ─────⎟  ⎥
⎢╲╱      ⎝  136    136 ⎠        ╲╱          ⎝136    136 ⎠  ⎥
⎢                                                          ⎥
⎢         21    √9689                   21    √9689        ⎥
⎢         ─── - ─────                   ─── + ─────        ⎥
⎢         136    136                    136    136         ⎥
⎢─────────────────────────────  ───────────────────────────⎥
⎢     ________________________       ______________________⎥

In [ ]:
img = np.array(Image.open("test_cat.png").convert("RGB"), dtype=np.uint8)

plt.imshow(img)
plt.title("Original Image")
plt.axis("off")
plt.show()

In [ ]:
m, n, c = img.shape

max_k = min(m, n)

k_values = []
mse_values = []
psnr_values = []
cr_values = []

# Sample k values
k_range = range(1, max_k + 1, 5)

for k in k_range:

    compressed_img = compress_channel(img, k)

    # Storage calculation
    num_values_per_channel = k * (m + n + 1)
    total_values = 3 * num_values_per_channel

    # Compression ratio
    cr = img.size / total_values

    # MSE
    mse = np.mean(
        (img.astype(np.float64) - compressed_img.astype(np.float64)) ** 2
    )

    # PSNR
    if mse == 0:
        psnr = np.inf
    else:
        psnr = 20 * np.log10(255 / np.sqrt(mse))

    # Store
    k_values.append(k)
    mse_values.append(mse)
    psnr_values.append(psnr)
    cr_values.append(cr)

In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(k_values, mse_values)

plt.xlabel("k (Number of Singular Values)")
plt.ylabel("MSE")
plt.title("k vs Mean Squared Error")

plt.yscale("log")

plt.grid(True)
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(k_values, psnr_values)

plt.xlabel("k (Number of Singular Values)")
plt.ylabel("PSNR (dB)")
plt.title("k vs PSNR")

plt.grid(True)
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(k_values, cr_values)

plt.xlabel("k (Number of Singular Values)")
plt.ylabel("Compression Ratio")
plt.title("k vs Compression Ratio")

plt.yscale("log")

plt.grid(True)
plt.show()

In [ ]:
R = img[:, :, 0].astype(float)

U, S, Vt = np.linalg.svd(R, full_matrices=False)

plt.figure(figsize=(8,5))

plt.plot(S)
plt.yscale("log")

plt.xlabel("Index")
plt.ylabel("Singular Value")
plt.title("Singular Value Spectrum")

plt.grid(True)
plt.show()

In [ ]:
energy = np.cumsum(S**2) / np.sum(S**2)

plt.figure(figsize=(8,5))
plt.plot(energy)

plt.xlabel("k")
plt.ylabel("Cumulative Energy")
plt.title("Energy Captured by First k Singular Values")

plt.grid(True)
plt.show()